# ResNet‑18 Image Embedding Pipeline (Full Visualization)

This notebook demonstrates the *complete pipeline* for converting an image into a **512‑D embedding** using a pretrained **ResNet‑18**:

- Layer‑wise feature‑map extraction
- Visualization of convolution blocks
- Global Average Pooling
- Final embedding vector
- Export of embeddings as CSV


## Imports

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math
from torchvision import models, transforms
from PIL import Image
from pathlib import Path
from matplotlib.patches import FancyArrowPatch


## Device (CPU‑only, Binder compatible)

In [ ]:
device = torch.device("cpu")
print("Using device:", device)

## Load ResNet‑18 in embedding mode
The classification head is removed so the model outputs a **512‑D feature vector**.

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Identity()
model.eval()
print("ResNet‑18 loaded in embedding mode")

## Store feature maps using forward hooks

In [ ]:
feature_maps = {}

def hook_fn(name):
    def hook(module, input, output):
        feature_maps[name] = output.detach().cpu()
    return hook

model.conv1.register_forward_hook(hook_fn("conv1"))
model.layer1.register_forward_hook(hook_fn("layer1"))
model.layer2.register_forward_hook(hook_fn("layer2"))
model.layer3.register_forward_hook(hook_fn("layer3"))
model.layer4.register_forward_hook(hook_fn("layer4"))

print("Feature‑map hooks registered")

## Image preprocessing (ImageNet, no padding)

In [ ]:
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(256),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

## Load image from GitHub data folder

In [ ]:
img_path = Path("data") / "25DIV3FAY_1002.jpg"
assert img_path.exists(), f"Image not found: {img_path}"

img = Image.open(img_path).convert("RGB")
x = transform(img).unsqueeze(0).to(device)

plt.figure(figsize=(4,4))
plt.imshow(img)
plt.axis("off")
plt.title("Input Image")
plt.show()

## Forward pass (populate feature maps)

In [ ]:
with torch.no_grad():
    embedding = model(x).squeeze().cpu().numpy()

print("Embedding shape:", embedding.shape)

## Create embedding DataFrame

In [ ]:
df_embedding = pd.DataFrame([
    embedding
], columns=[f"emb_{i}" for i in range(len(embedding))])

df_embedding.insert(0, "image_id", img_path.name)
df_embedding

## Save embedding to CSV (downloadable)

In [ ]:
csv_path = "image_embedding.csv"
df_embedding.to_csv(csv_path, index=False)
print(f"✅ Embedding saved to {csv_path}")

## Full ResNet‑18 architecture diagram with feature maps

In [ ]:
layer_order = [
    "conv1",
    "layer1.0.conv1", "layer1.0.conv2",
    "layer1.1.conv1", "layer1.1.conv2",
    "layer2.0.conv1", "layer2.0.conv2",
    "layer2.1.conv1", "layer2.1.conv2",
    "layer3.0.conv1", "layer3.0.conv2",
    "layer3.1.conv1", "layer3.1.conv2",
    "layer4.0.conv1", "layer4.0.conv2",
    "layer4.1.conv1", "layer4.1.conv2",
]

def draw_arrow(ax, x1, y1, x2, y2):
    ax.add_patch(FancyArrowPatch((x1,y1),(x2,y2),arrowstyle="->",linewidth=1))

def representative_maps(fmap, n=3):
    fmap = fmap.squeeze(0)
    C = fmap.shape[0]
    idxs = np.linspace(0, C-1, n, dtype=int)
    maps = []
    for i in idxs:
        fm = fmap[i]
        fm = (fm - fm.min()) / (fm.max() - fm.min() + 1e-6)
        maps.append(fm.numpy())
    return maps

fig, ax = plt.subplots(figsize=(22,7))
ax.axis("off")
ax.set_xlim(0, 30)
ax.set_ylim(0, 8)

ax.imshow(img.resize((128,128)), extent=(0.5,2.5,4,6))
ax.text(1.5,3.5,"Input Image\n3×256×256",ha="center")

prev_x, prev_y = 2.5, 5.0
x = 4.0

for lname in layer_order:
    fmap = feature_maps[lname.split('.')[0]] if lname != 'conv1' else feature_maps['conv1']
    reps = representative_maps(fmap)
    C,H,W = fmap.shape[1:]
    for i,fm in enumerate(reps):
        ax.imshow(fm, extent=(x,x+1,1.5+i*1.1,2.5+i*1.1), cmap='viridis')
    ax.text(x+0.5,1.1,f"{lname}\n{C}×{H}×{W}",ha='center',fontsize=7)
    draw_arrow(ax, prev_x, prev_y, x, 3.5)
    prev_x, prev_y = x+1, 3.5
    x += 1.4

draw_arrow(ax, prev_x, prev_y, x, prev_y)
ax.add_patch(plt.Rectangle((x,3),1.6,1,fc='#dddddd',ec='black'))
ax.text(x+0.8,2.4,"Global Avg Pool\n512",ha='center')

draw_arrow(ax, x+1.6, prev_y, x+3.6, prev_y)
ax.add_patch(plt.Rectangle((x+3.6,3),3,1,fc='#cccccc',ec='black'))
ax.text(x+5.1,2.4,"Image Embedding\n512‑D",ha='center')

plt.suptitle("ResNet‑18 Image Embedding Pipeline",fontsize=14)
plt.tight_layout()
plt.show()